In [1]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

In [2]:
# Import helpers
from helpers.validation_helpers.filtering_helper import *
from helpers.validation_helpers.session_length_helper import *
from helpers.validation_helpers.location_score_function import *
from helpers.heatmap_building.heatmap_builder import *

In [3]:
sys.path.append(
    os.path.abspath("helpers/Protobuffer_Deserialization")
)

In [4]:
from helpers.Protobuffer_Deserialization.decoding import *

In [5]:
# Read events data
df1 = pd.read_csv("data/customer1-events-1-2025-to-3-2025.csv")
df2 = pd.read_csv("data/customer2-events-1-2025-to-3-2025.csv")
df_pandas = pd.concat([df1, df2], ignore_index=True)

/var/folders/_5/zwklnk315yqbrfhydp3_pj5w0000gn/T/ipykernel_38604/4062907009.py:2: DtypeWarning: Columns (7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv("data/customer1-events-1-2025-to-3-2025.csv")
/var/folders/_5/zwklnk315yqbrfhydp3_pj5w0000gn/T/ipykernel_38604/4062907009.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("data/customer2-events-1-2025-to-3-2025.csv")


In [6]:
# Filter for normal -> off or normal -> charging modes only
events = filter_correct_modes(df_pandas)

In [7]:
# Extract latitude/longitudes with Protobuffer deserialization
mapped_events = deserialize_events(events)

2026-04-29 02:11:15.450758 Starting decoding...
2026-04-29 02:12:15.888862 Completed decoding.
2026-04-29 02:12:24.658465 Starting lat/lon extraction...
2026-04-29 02:13:39.423849 Completed lat/lon extraction.


In [8]:
# Filter for sessions with at least one target alarm
alarm_events = filter_target_alarm_sessions(mapped_events)

In [9]:
# Calculate location scores
scored_events = location_validation(alarm_events)

In [10]:
# Sessions that appear in scored_events
scored_sessions = set(scored_events["SESSION_ID"].unique())

# From events: keep only sessions not in scored_events,
# and within those sessions keep only LOCATION rows
mapped_events_only = mapped_events[
    (~mapped_events["SESSION_ID"].isin(scored_sessions))
    & (mapped_events["EVENT_TYPE"] == "LOCATION")
].copy()

# From scored_events: keep only ALARM and LOCATION rows
scored_keep = scored_events[
    scored_events["EVENT_TYPE"].isin(["ALARM", "LOCATION"])
].copy()

# Combine them, preserving the extra columns from scored_events
complete_heatmap_data = pd.concat([scored_keep, mapped_events_only], ignore_index=True, sort=False)

In [11]:
# final data
data = complete_heatmap_data
data['LOGGED_AT'] = pd.to_datetime(data['LOGGED_AT'], errors='coerce', format='mixed')

In [12]:
# Customer 1 Heatmap
m1 = generate_customer_heatmap(
    data=data,
    customer_id="1638b8ad-4e9d-4120-bbc2-27041d1ee1e5",
    output_file="alarm_heatmap_v9_c1.html",
    h3_res=9
)

Heatmap saved to: alarm_heatmap_v9_c1.html


In [13]:
# Customer 2 Heatmap
m2 = generate_customer_heatmap(
    data=data,
    customer_id="7e990120-e250-4559-a8ce-762e5c867c49",
    output_file="alarm_heatmap_v9_c2.html",
    h3_res=9
)

Heatmap saved to: alarm_heatmap_v9_c2.html
